## Building a logistic regression model



In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [3]:
# Read data in and view first few entries
df = pd.read_csv('https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Data/classification_sprint/claims_data.csv')
df

,age,sex,bmi,steps,children,smoker,region,insurance_claim,claim_amount
0,19,female,27.900,3009,0,yes,southwest,yes,16884.9240
1,18,male,33.770,3008,1,no,southeast,yes,1725.5523
2,28,male,33.000,3009,3,no,southeast,no,0.0000
3,33,male,22.705,10009,0,no,northwest,no,0.0000
4,32,male,28.880,8010,0,no,northwest,yes,3866.8552
...,...,...,...,...,...,...,...,...,...
1333,50,male,30.970,4008,3,no,northwest,no,0.0000
1334,18,female,31.920,3003,0,no,northeast,yes,2205.9808
1335,18,female,36.850,3008,0,no,southeast,yes,1629.8335
1336,21,female,25.800,8009,0,no,southwest,no,0.0000


### Preprocessing

We will start by preparing the data to be used in the logistic regression algorithm. This involves:

- Splitting the data into features and labels.
- Transforming the categorical features (create dummy variables).
- Splitting the data into training and testing sets.

In [7]:
# labels
y = df['insurance_claim']

# features
X = df.drop('insurance_claim', axis=1)
print(X)

      age     sex     bmi  steps  children smoker     region  claim_amount
0      19  female  27.900   3009         0    yes  southwest    16884.9240
1      18    male  33.770   3008         1     no  southeast     1725.5523
2      28    male  33.000   3009         3     no  southeast        0.0000
3      33    male  22.705  10009         0     no  northwest        0.0000
4      32    male  28.880   8010         0     no  northwest     3866.8552
...   ...     ...     ...    ...       ...    ...        ...           ...
1333   50    male  30.970   4008         3     no  northwest        0.0000
1334   18  female  31.920   3003         0     no  northeast     2205.9808
1335   18  female  36.850   3008         0     no  southeast     1629.8335
1336   21  female  25.800   8009         0     no  southwest        0.0000
1337   61  female  29.070   8008         0    yes  northwest    29141.3603

[1338 rows x 8 columns]


In [9]:
# Transforming the Features
X_transformed = pd.get_dummies(X, drop_first=True)
print(X_transformed)

      age     bmi  steps  children  claim_amount  sex_male  smoker_yes  \
0      19  27.900   3009         0    16884.9240     False        True   
1      18  33.770   3008         1     1725.5523      True       False   
2      28  33.000   3009         3        0.0000      True       False   
3      33  22.705  10009         0        0.0000      True       False   
4      32  28.880   8010         0     3866.8552      True       False   
...   ...     ...    ...       ...           ...       ...         ...   
1333   50  30.970   4008         3        0.0000      True       False   
1334   18  31.920   3003         0     2205.9808     False       False   
1335   18  36.850   3008         0     1629.8335     False       False   
1336   21  25.800   8009         0        0.0000     False       False   
1337   61  29.070   8008         0    29141.3603     False        True   

      region_northwest  region_southeast  region_southwest  
0                False             False          

In [10]:
from sklearn.model_selection import train_test_split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X_transformed, y, test_size=0.2, random_state=50)

Our data is now ready. Let's train the logistic regression model.

### Training

We need the LogisticRegression module from `sklearn.linear_model`.

In [12]:
from sklearn.linear_model import LogisticRegression

We create an instance of the `LogisticRegression()` object using the default parameters.

In [13]:
lr = LogisticRegression()

We use the `fit()` method to train the model.

**Pro tip**: In the multi-class case we referred to above, the `LogisticRegression` instance takes a simple argument which enables it to be used for 2+ classes: `multi_class='ovr'`. We'll revisit this later in the course.

In [14]:
lr.fit(X_train, y_train)

LogisticRegression()

Now that the model is trained, we can extract the parameters. The parameters consist of the intercept and the coefficients related to the features. These parameters can be used to predict future claims given the features.

#### Intercept

The interpretation of the parameters of the logistic model is not quite the same as for a linear regression model.

In binary classification, the class with value 1 is known as the _reference class_. Let's explore.

The intercept, $\beta_0$, is interpreted as the **log odds** ratio of an observation being in the reference class when all other predictor variables are equal to zero.

We can exponentiate this value, i.e. raise the natural number $e$ to this value to convert it to a typical **odds** ratio. In other words:

$$Odds = e^{\beta_0}$$

In [15]:
lr.intercept_[0]

np.float64(-1.126111975327057e-06)

#### Coefficients

For binary categorical variables, like `smoker` and `sex`, the coefficient is interpreted as the **log odds** ratio between the class implied by a zero for the variable (i.e. non-smoker), and the class implied by a one for the variable (i.e. smoker).

For continuous variables, the coefficient is interpreted as the expected change in the log odds for a one-unit increase in the variable.

Again, we can arrive at the usual odds value by exponentiating the coefficient:

$$Odds = e^{\beta_1}$$

Effectively, each coefficient is a measure of the change in the log odds of belonging to the reference class for one-unit changes in the variable.

In [16]:
coeff_df = pd.DataFrame(lr.coef_.T, X_transformed.columns, columns=['Coefficient'])
coeff_df

,Coefficient
age,-2.674690e-05
bmi,-1.715059e-05
steps,-9.661165e-03
children,-1.091721e-05
claim_amount,9.938677e-02
sex_male,-2.057983e-07
smoker_yes,1.188968e-06
region_northwest,-9.551386e-07
region_southeast,6.362230e-07
region_southwest,-6.313046e-07


In [17]:
import numpy as np

intercept = lr.intercept_[0]
print(f"\nIntercept: {intercept:.7f}")

print("\nCoefficients:")
print(coeff_df)

print("\n--- Interpretation of Intercept ---")
print(f"The intercept (approximately {intercept:.7f}) represents the log odds of an insurance claim when all other predictor variables are zero. Exponentiating this value (e^({intercept:.7f}) = {np.exp(intercept):.7f}) gives the odds ratio. Since the intercept is very close to zero, it suggests that with all other features at zero, the odds of an insurance claim are very close to 1.")

print("\n--- Interpretation of Coefficients ---")
print("Each coefficient represents the change in the log odds of an insurance claim for a one-unit increase in the corresponding feature, assuming all other features are held constant. To get the odds ratio, you can exponentiate the coefficient (e^coefficient).")
print("\nHere are some specific interpretations:")

# Interpret claim_amount
coeff_claim_amount = coeff_df.loc['claim_amount', 'Coefficient']
print(f"- **Claim Amount ({coeff_claim_amount:.7f})**: For every one-unit increase in 'claim_amount', the log odds of an insurance claim increases by approximately {coeff_claim_amount:.7f}. This means higher claim amounts are positively associated with making an insurance claim (e^{coeff_claim_amount:.7f} = {np.exp(coeff_claim_amount):.7f} odds ratio).")

# Interpret steps
coeff_steps = coeff_df.loc['steps', 'Coefficient']
print(f"- **Steps ({coeff_steps:.7f})**: For every one-unit increase in 'steps', the log odds of an insurance claim decreases by approximately {coeff_steps:.7f}. This suggests that individuals who take more steps are slightly less likely to make an insurance claim (e^{coeff_steps:.7f} = {np.exp(coeff_steps):.7f} odds ratio).")

# Interpret smoker_yes
coeff_smoker_yes = coeff_df.loc['smoker_yes', 'Coefficient']
print(f"- **Smoker_yes ({coeff_smoker_yes:.7f})**: Being a smoker (compared to a non-smoker) is associated with a very slight increase in the log odds of an insurance claim ({coeff_smoker_yes:.7f}). This implies a negligible positive association with making a claim (e^{coeff_smoker_yes:.7f} = {np.exp(coeff_smoker_yes):.7f} odds ratio).")

# Interpret sex_male
coeff_sex_male = coeff_df.loc['sex_male', 'Coefficient']
print(f"- **Sex_male ({coeff_sex_male:.7f})**: Being male (compared to female) is associated with a very slight decrease in the log odds of an insurance claim ({coeff_sex_male:.7f}). This implies a negligible negative association with making a claim (e^{coeff_sex_male:.7f} = {np.exp(coeff_sex_male):.7f} odds ratio).")

print("\n- **Other variables (age, bmi, children, region_northwest, region_southeast, region_southwest)**: Most other coefficients are very close to zero, indicating that these features have a negligible linear relationship with the log odds of making an insurance claim in this model.")



Intercept: -0.0000011

Coefficients:
                   Coefficient
age              -2.674690e-05
bmi              -1.715059e-05
steps            -9.661165e-03
children         -1.091721e-05
claim_amount      9.938677e-02
sex_male         -2.057983e-07
smoker_yes        1.188968e-06
region_northwest -9.551386e-07
region_southeast  6.362230e-07
region_southwest -6.313046e-07

--- Interpretation of Intercept ---
The intercept (approximately -0.0000011) represents the log odds of an insurance claim when all other predictor variables are zero. Exponentiating this value (e^(-0.0000011) = 0.9999989) gives the odds ratio. Since the intercept is very close to zero, it suggests that with all other features at zero, the odds of an insurance claim are very close to 1.

--- Interpretation of Coefficients ---
Each coefficient represents the change in the log odds of an insurance claim for a one-unit increase in the corresponding feature, assuming all other features are held constant. To get the o

### Prediction time

Next we will use the `predict` method to obtain predictions from our test data observations.

In [18]:
pred_lr = lr.predict(X_test)